In [ ]:
!pip install -q torch transformers datasets scikit-learn onnx onnxruntime accelerate

In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
import json
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import f1_score, precision_recall_fscore_support

LABEL_LIST = [
    "All-or-Nothing Thinking",
    "Overgeneralization",
    "Mental Filter",
    "Disqualifying the Positive",
    "Mind Reading",
    "Fortune-Telling",
    "Catastrophizing",
    "Emotional Reasoning",
    "Should Statements",
    "Labeling",
    "Personalization",
    "Blame"
]

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
raw_datasets = load_dataset("masked-kunsiquat/shreevastava-cognitive-distortions")

if "validation" not in raw_datasets and "test" in raw_datasets:
    dataset_splits = raw_datasets
elif "train" in raw_datasets and "test" not in raw_datasets:
    split_data = raw_datasets["train"].train_test_split(test_size=0.15, seed=42)
else:
    dataset_splits = raw_datasets

train_data = dataset_splits["train"]
eval_key = "validation" if "validation" in dataset_splits else "test"
eval_data = dataset_splits[eval_key]

In [ ]:
def preprocess_function(examples):
    tokenized = tokenizer(
        examples["text"],
        max_length=128,
        truncation=True,
        padding=False
    )
    labels_matrix = []
    num_samples = len(examples["text"])
    for i in range(num_samples):
        row = []
        for label_name in LABEL_LIST:
            val = 0
            if label_name in examples:
                val = int(examples[label_name][i])
            elif "labels" in examples and isinstance(examples["labels"][i], list):
                val = 1 if label_name in examples["labels"][i] else 0
            row.append(val)
        labels_matrix.append(row)
    tokenized["labels"] = labels_matrix
    return tokenized

tokenized_train = train_data.map(preprocess_function, batched=True, remove_columns=train_data.column_names)
tokenized_eval = eval_data.map(preprocess_function, batched=True, remove_columns=eval_data.column_names)

all_train_labels = np.array(tokenized_train["labels"])
pos_counts = np.sum(all_train_labels, axis=0)
total_samples = len(all_train_labels)
neg_counts = total_samples - pos_counts
pos_weights = np.where(pos_counts > 0, neg_counts / (pos_counts + 1e-5), 1.0)
pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float)

In [ ]:
class MultiLabelTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight
        if self.pos_weight is not None:
            self.loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)
        else:
            self.loss_fn = torch.nn.BCEWithLogitsLoss()

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        if self.pos_weight is not None and self.pos_weight.device != logits.device:
            self.loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(logits.device))
        loss = self.loss_fn(logits, labels.float())
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1.0 / (1.0 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    return {"macro_f1": float(macro_f1), "micro_f1": float(micro_f1)}

id2label = {idx: label for idx, label in enumerate(LABEL_LIST)}
label2id = {label: idx for idx, label in enumerate(LABEL_LIST)}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(LABEL_LIST),
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=50,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=20,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = MultiLabelTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    pos_weight=pos_weight_tensor
)

trainer.train()
trainer.save_model("./distilbert_saved")
tokenizer.save_pretrained("./distilbert_saved")

In [ ]:
eval_raw = load_dataset("joyboseroy/CognitiveDistortion-Eval", split="train")
eval_texts = eval_raw["text"]
eval_labels = []
for i in range(len(eval_texts)):
    row = []
    for label_name in LABEL_LIST:
        val = 0
        if label_name in eval_raw.column_names:
            val = int(eval_raw[label_name][i])
        elif "labels" in eval_raw.column_names and isinstance(eval_raw["labels"][i], list):
            val = 1 if label_name in eval_raw["labels"][i] else 0
        row.append(val)
    eval_labels.append(row)
targets = np.array(eval_labels)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

all_probs = []
with torch.no_grad():
    for i in range(0, len(eval_texts), 32):
        batch = eval_texts[i : i + 32]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        logits = model(**enc).logits.detach().cpu().numpy()
        probs = 1.0 / (1.0 + np.exp(-logits))
        all_probs.append(probs)
eval_probs = np.vstack(all_probs)

best_thresholds = np.full(eval_probs.shape[1], 0.5)
for class_idx in range(eval_probs.shape[1]):
    best_f1 = -1.0
    y_true = targets[:, class_idx]
    if np.sum(y_true) == 0:
        continue
    for th in np.linspace(0.1, 0.9, 17):
        y_pred = (eval_probs[:, class_idx] >= th).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_thresholds[class_idx] = th

tuned_preds = (eval_probs >= best_thresholds).astype(int)
print("Macro F1 (Tuned):", f1_score(targets, tuned_preds, average="macro", zero_division=0))
print("Micro F1 (Tuned):", f1_score(targets, tuned_preds, average="micro", zero_division=0))

In [ ]:
import os
import onnx
import onnxruntime as ort

os.makedirs("./artifacts/tokenizer", exist_ok=True)
tokenizer.save_pretrained("./artifacts/tokenizer")

dummy_text = "I always fail at everything I try"
dummy_inputs = tokenizer(dummy_text, return_tensors="pt", padding="max_length", max_length=128, truncation=True).to("cpu")
model.to("cpu")
model.eval()

onnx_path = "./artifacts/model.onnx"
torch.onnx.export(
    model,
    (dummy_inputs["input_ids"], dummy_inputs["attention_mask"]),
    onnx_path,
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "attention_mask": {0: "batch_size", 1: "sequence_length"},
        "logits": {0: "batch_size"}
    },
    opset_version=14,
    do_constant_folding=True
)

threshold_map = {LABEL_LIST[i]: float(best_thresholds[i]) for i in range(len(LABEL_LIST))}
with open("./artifacts/tuned_thresholds.json", "w", encoding="utf-8") as f:
    json.dump(threshold_map, f, indent=2)

!zip -r furrmind_model_artifacts.zip artifacts/
print("Finished. Ready to download furrmind_model_artifacts.zip")